Directions:
1) Run All and wait a bit for ultralytics, its dependencies and files
2) Drag and drop video files (avi, mp4, mov) into content->ultralytics->ultralytics->videos
3) Run last cell to use yolo player detection on all files (~1-2 seconds per frame)
4) View detected files in content->ultralytics->ultralytics->all_videos_output

In [ ]:
!git clone https://github.com/ultralytics/ultralytics.git

Cloning into 'ultralytics'...
remote: Enumerating objects: 89347, done.
remote: Counting objects: 100% (1246/1246), done.
remote: Compressing objects: 100% (399/399), done.
remote: Total 89347 (delta 1145), reused 847 (delta 847), pack-reused 88101 (from 4)
Receiving objects: 100% (89347/89347), 48.73 MiB | 9.03 MiB/s, done.
Resolving deltas: 100% (67244/67244), done.


In [ ]:
%pip install ultralytics
import ultralytics
# %pip uninstall torch torchvision torchaudio
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
dependencies = [
    "numpy",
    "matplotlib",
    "opencv-python",
    "pillow",
    "pyyaml",
    "requests",
    "scipy",
    "torch",
    "torchvision",
    "psutil",
    "polars",
    "ultralytics-thop",
]

for dep in dependencies:
    print(f"Checking {dep}:")
    !pip show {dep}
    print("-" * 20)

Checking numpy:
Name: numpy
Version: 2.0.2
Summary: Fundamental package for array computing in Python
Home-page: https://numpy.org
Author: Travis E. Oliphant et al.
Author-email: 
License: Copyright (c) 2005-2024, NumPy Developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are
met:

    * Redistributions of source code must retain the above copyright
       notice, this list of conditions and the following disclaimer.

    * Redistributions in binary form must reproduce the above
       copyright notice, this list of conditions and the following
       disclaimer in the documentation and/or other materials provided
       with the distribution.

    * Neither the name of the NumPy Developers nor the names of any
       contributors may be used to endorse or promote products derived
       from this software without specific prior written permission.

THIS SOFTWARE IS PROVID

In [ ]:
import os

# create videos folder to store data.
folder_path = '/content/ultralytics/ultralytics/videos'

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_path}' created.")
else:
    print(f"Folder '{folder_path}' already exists.")

Folder '/content/ultralytics/ultralytics/videos' created.


3/12 Changes:

Run cell below for detections. same instructions as the top description.
- conf = 0.35-0.45, iou = 0.8
- swapped from model.predict to model.track to be able to use bytetrack.yaml for "better" detections
- added batch & workers attribute to have more consistent run times. (detecting X frames at a time)
- video clip for comparison is 14 seconds long.
- Used different ranges of imgsz (image sizes) for detection. [sz640 = ~440 seconds, sz448 = ~382 seconds, sz416 = ~356 seconds, sz320 = ~220 seconds]
- lowering image size lowers run time, but lowers accuracy.
- Personally, sz416 works the best out of all of this so far. It was at sz640 during previous iterations (~10-25 min w/o applied methods). (except for the ball x player one sz992)

In [ ]:
from ultralytics import YOLO
import os
import time


# Load a pre-trained YOLO model ('yolov8s.pt' isnt good. get an upgrade [s, m, l])
model = YOLO('yolov8m.pt')
# model.to("cuda")

# Folder containing videos
video_folder = '/content/ultralytics/ultralytics/videos' # input folder (put videos here)
output_folder = '/content/ultralytics/ultralytics/all_videos_output' # output folder (view videos with detetections here)
os.makedirs(output_folder, exist_ok=True)

# Get list of video files
video_files = [
    os.path.join(video_folder, f)
    for f in os.listdir(video_folder)
    if f.endswith(('.mp4', '.avi', '.mov'))
]

# Check if any videos were found
if not video_files:
    print(f"No video files found in {video_folder}")
else:
    print(f"Found {len(video_files)} videos to look at.")

# instantiate time (this is to show how long this process takes)
start_time = time.time()
# Process each video file individually and save results to a single folder (gonna take rlly long maybe find a way to run them in multiple processes?)
for video_index, video_path in enumerate(video_files):
    print(f"Processing video {video_index + 1}/{len(video_files)}: {os.path.basename(video_path)}")

    # Run YOLOv8 on the current video
    # conf = will box the detected players if they are X% confident.
    # iou = if boxes overlap by X%, keep the one with the highest confidence (maybe set it high cause some sports require close contact)
    results = model.track( # replace predict with track
        source=video_path,
        classes=[0], #class[0] = people
        stream=True,
        imgsz=416, # try lowering image size for detection a bit? faster but tanks accuracy
        save=True,
        persist=True,
        conf=0.35, # get the right balance of being able to track players accurately whilst avoiding spectators on the side
        iou=0.8, # if the system thinks it's X percent confident that 2 detected are collided, detect the highest of the two.
        project=output_folder,
        name='',  # All outputs go into the same folder (name above)
        exist_ok=True,
        tracker="bytetrack.yaml", # ultralytics alr has bytetrack.
        batch=16, # run X detections at a time. allows for consistant run times.
        cache=True,
        workers=24, # not sure why but as long as workers >= batch, it makes running speed faster.
        # device=0,
        # vid_stride=4, # process every X seconds. makes it X times faster.
        # persist=True # make sure we keep the changes of vid_stride or else video will speed up X times.
    )

    # Iterate through results (one per frame)
    for i, result in enumerate(results):
        detections = result.boxes
        num_people = len(detections)
        print(f"  Video {video_index + 1}, Frame {i+1}: Detected {num_people} people")

end_time = time.time()
total_time = end_time - start_time
print(f"Total time taken: {total_time} second(s) ({total_time/60} minute(s)) to view {len(video_files)} files.")

Found 1 videos to look at.
Processing video 1/1: soccer2.mp4

video 1/1 (frame 1/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 7 persons, 425.3ms
video 1/1 (frame 2/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 6 persons, 425.3ms
video 1/1 (frame 3/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 7 persons, 425.3ms
video 1/1 (frame 4/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 7 persons, 425.3ms
video 1/1 (frame 5/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 7 persons, 425.3ms
video 1/1 (frame 6/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 9 persons, 425.3ms
video 1/1 (frame 7/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 9 persons, 425.3ms
video 1/1 (frame 8/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 9 persons, 425.3ms
video 1/1 (frame 9/713) /content/ultralytics/ultralytics/videos/soccer2.mp4: 256x416 9 persons, 425.3ms
vi